# IDPFold2 Monomer Preview (Colab)

This notebook installs dependencies, downloads checkpoint, runs monomer inference, and previews `.pdb` in 3D.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/IDPFold-multimer')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Junjie-Zhu/IDPFold2', str(REPO_DIR)], check=True)

subprocess.run(['pip', 'install', '-q', 'fair-esm', 'py3Dmol', 'ipywidgets'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
print('Environment is ready')

In [ ]:
import urllib.request

CHECKPOINT_URL = 'https://zenodo.org/records/18239596/files/IDPFold2_ema_0.999_260114.pth?download=1'
CHECKPOINT_DIR = Path('/content/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = CHECKPOINT_DIR / 'IDPFold2_ema_0.999_260114.pth'

if not CKPT_PATH.exists():
    print('Downloading checkpoint...')
    urllib.request.urlretrieve(CHECKPOINT_URL, CKPT_PATH)

if not CKPT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint download failed: {CKPT_PATH}')

print('Checkpoint:', CKPT_PATH)

In [ ]:
import pandas as pd

# @param ['example', 'custom_sequence']
INPUT_MODE = 'example'

# used when INPUT_MODE='custom_sequence'
CUSTOM_TEST_CASE = 'custom_demo'
CUSTOM_SEQUENCE = 'GPGSEDVWEILRQAPPSEYERIAFQYGVTDLRGMLKRLKGMRRDEKKSTAFQKKLEPAYQVSKGHKIRLTVELADHDAEVKWLKNGQEIQMSGSKYIFESIGAKRTLTISQCSLADDAAYQCVVGGEKCSTELFVKE'

WORK_DIR = REPO_DIR
INPUT_CSV = Path('/content/input_monomer.csv')

if INPUT_MODE == 'example':
    INPUT_CSV = WORK_DIR / 'data' / 'monomer_example.csv'
else:
    pd.DataFrame([{'test_case': CUSTOM_TEST_CASE, 'sequence': CUSTOM_SEQUENCE}]).to_csv(INPUT_CSV, index=False)

print('Input CSV:', INPUT_CSV)

In [ ]:
import glob
import shlex

PREFIX = 'COLAB_MONOMER'  # @param {type:'string'}
NSAMPLES = 4              # @param {type:'integer'}
MAX_BATCH_LENGTH = 3500   # @param {type:'integer'}

PLM_EMB_DIR = Path('/content/embeddings')
LOGGING_DIR = Path('/content/outputs')
PLM_EMB_DIR.mkdir(parents=True, exist_ok=True)
LOGGING_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    'python', str(WORK_DIR / 'src' / 'inference.py'),
    f'prefix={PREFIX}',
    f'ckpt_dir={CKPT_PATH}',
    f'plm_emb_dir={PLM_EMB_DIR}',
    f'csv_dir={INPUT_CSV}',
    f'nsamples={NSAMPLES}',
    f'max_batch_length={MAX_BATCH_LENGTH}',
    f'logging_dir={LOGGING_DIR}',
]

print('Running:', ' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True, cwd=WORK_DIR)

pdb_files = sorted(glob.glob(str(LOGGING_DIR / '**' / 'samples' / '*.pdb'), recursive=True))
if not pdb_files:
    raise FileNotFoundError('No PDB file was generated.')
PDB_PATH = Path(pdb_files[-1])
print('Generated PDB:', PDB_PATH)

In [ ]:
import py3Dmol
from google.colab import files

pdb_text = PDB_PATH.read_text()
viewer = py3Dmol.view(width=900, height=600)
viewer.addModel(pdb_text, 'pdb')
viewer.setStyle({'cartoon': {'color': 'spectrum'}})
viewer.zoomTo()
viewer.show()

print(f'Download file: {PDB_PATH}')
files.download(str(PDB_PATH))